<a href="https://colab.research.google.com/github/humaaslam46/flyRank-ml-Internship-tasks/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Internship-ml"):
        os.system("git clone --depth 1 https://github.com/humaaslam46/Internship-ml")
    os.chdir("Internship-ml")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
features = ["impressions_90d","days_since_last_update","avg_position","ctr","word_count","engagement_rate","content_age_days","search_volume"]
print("Ready:", df.shape)

Ready: (30000, 45)


## 1. Two paper findings + my methodology questions

Finding #1 (paper): "The Anatomy of Growing Content" - growing pages average
37.6% more words and are 20% younger than declining pages (74.8K growing vs
45.6K declining pages).
My methodology question: this is explicitly framed as observational, which
the paper itself is honest about - but the causal direction is still
ambiguous. Are longer, younger pages growing because of length/freshness,
or do already-growing pages simply get expanded more, since success invites
more editorial investment (reverse causality)? Also worth asking: is "trend
direction" here computed the same way as my own is_declining proxy
(30-day-vs-prior-30-day), which would mean it inherits the same
current-window-proxy limitation I've flagged in my own work.

Finding #2 (paper): "The Content Performance Curve" - health score peaks at
61-90 days, drops to a cliff by 271-365 days, then partially recovers at
365+ days, attributed to "concentrated in older pages that were refreshed."
My methodology question: was refresh status actually stratified within the
365+ bucket (refreshed vs non-refreshed pages in that same age range), or is
this an aggregate average with an interpretive story layered on top? Without
that split shown, "recovery is due to refresh" reads as a plausible
narrative rather than a demonstrated result.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_split(train, test, label):
    Xtr = train[features].replace([np.inf,-np.inf],np.nan).fillna(0)
    Xte = test[features].replace([np.inf,-np.inf],np.nan).fillna(0)
    ytr, yte = train["is_declining"], test["is_declining"]
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    p50 = precision_at_k(rf.predict_proba(Xte)[:,1], yte.values, 50)
    overlap = len(set(train["client_id"]) & set(test["client_id"]))
    print(f"{label}: Precision@50={p50:.3f}, client overlap={overlap}")
    return p50

train_naive, test_naive = train_test_split(df, test_size=0.25, random_state=42, stratify=df["is_declining"])
p_naive = run_split(train_naive, test_naive, "BEFORE - naive random split")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_grp, test_grp = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
p_grouped = run_split(train_grp, test_grp, "AFTER - grouped split")

print(f"Inflation from leakage: {(p_naive/p_grouped - 1)*100:.1f}%")

BEFORE - naive random split: Precision@50=0.860, client overlap=31
AFTER - grouped split: Precision@50=0.540, client overlap=0
Inflation from leakage: 59.3%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
for f in features:
    print(f, round(df[f].corr(df["is_declining"]), 3))
print("\nNo feature shows a suspiciously strong correlation (max |0.164|).")
print("Confirmed: none of these features are derived from trend_pct or trend_direction (the label source).")

impressions_90d -0.018
days_since_last_update 0.081
avg_position -0.029
ctr -0.062
word_count 0.09
engagement_rate -0.013
content_age_days -0.164
search_volume -0.019

No feature shows a suspiciously strong correlation (max |0.164|).
Confirmed: none of these features are derived from trend_pct or trend_direction (the label source).


## 4. Claim rewrite

Before: "Random Forest beat the baseline rule by 1.42x."
After: "On a client-grouped, leakage-checked split, the Random Forest showed
an observed, directional Precision@50 improvement of 1.42x over the baseline
rule (0.380 to 0.540). This is decision-support evidence for prioritizing a
review queue - not a causal claim about refresh outcomes, and not a
guarantee it holds on unseen clients or future data."

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.